# Benchmark de Transformación: Tradicional (PyMuPDF) vs IA (Docling)

Este notebook compara dos enfoques fundamentalmente distintos para la extracción de datos de PDF:
1. **PyMuPDF (fitz)**: Un motor tradicional, extremadamente rápido, que extrae texto plano basándose en coordenadas y capas del documento.
2. **Docling (IBM)**: Un motor de nueva generación que utiliza modelos de IA para comprender la estructura, tablas y jerarquía, exportando a Markdown.

El objetivo es evaluar el balance entre **velocidad** (PyMuPDF) y **calidad estructural** (Docling) para el procesamiento de informes del Congreso.

In [2]:
import os
import time
import fitz  # PyMuPDF
from pathlib import Path
from docling.document_converter import DocumentConverter
from IPython.display import display, Markdown

## 1. Configuración
Define aquí la ruta de tu informe del Congreso.

In [3]:
# === CONFIGURACIÓN ===
PDF_PATH = os.path.join("inputs", "test_1995.pdf")  # test_2026.pdf tiene texto, test_1995.pdf parece ser una imagen/escaneo
OUTPUT_DIR = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Diagnóstico de ruta (Windows + Python + Docling/Torch)
if any(ord(c) > 127 for c in os.path.abspath(".")):
    print("⚠️ ADVERTENCIA: La ruta actual contiene caracteres no-ASCII (acentos, ñ, etc).")
    print("Esto puede causar fallos en Docling/PyTorch en Windows.")
    print("Se recomienda renombrar la carpeta 'Construcción' a 'Construccion'.\n")

## 2. Funciones de Extracción

In [4]:
def process_with_pymupdf(pdf_path, output_folder):
    print("🚀 Procesando con PyMuPDF (Método Tradicional)...\n")
    start_time = time.time()
    
    text_content = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text_content.append(page.get_text("text"))
    
    full_text = "\n".join(text_content)
    duration = time.time() - start_time
    
    pdf_name = Path(pdf_path).stem
    output_path = os.path.join(output_folder, f"{pdf_name}_salida_pymupdf.txt")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(full_text)
    
    print(f"✅ PyMuPDF finalizado en {duration:.4f}s")
    return full_text, duration

def process_with_docling(pdf_path, output_folder):
    print("🚀 Procesando con Docling (Método IA)...\n")
    converter = DocumentConverter()
    start_time = time.time()
    
    result = converter.convert(pdf_path)
    markdown_content = result.document.export_to_markdown()
    
    duration = time.time() - start_time
    
    pdf_name = Path(pdf_path).stem
    output_path = os.path.join(output_folder, f"{pdf_name}_salida_docling.md")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(markdown_content)
    
    print(f"✅ Docling finalizado en {duration:.2f}s")
    return markdown_content, duration

## 3. Ejecución del Benchmark
Compara el rendimiento y los resultados.

In [ ]:
text_py, time_py = "", 0
text_doc, time_doc = "", 0

if os.path.exists(PDF_PATH):
    # 1. PyMuPDF
    try:
        text_py, time_py = process_with_pymupdf(PDF_PATH, OUTPUT_DIR)
        if len(text_py.strip()) == 0:
            print("⚠️ PyMuPDF no extrajo texto. El PDF podría ser una imagen/escaneo.")
    except Exception as e:
        print(f"❌ Error en PyMuPDF: {e}")

    print("-" * 30)
    
    # 2. Docling
    try:
        text_doc, time_doc = process_with_docling(PDF_PATH, OUTPUT_DIR)
    except Exception as e:
        print(f"❌ Error en Docling: {e}")
        print("Tip: Verifica que no haya acentos en la ruta de las carpetas.")
    
    # Comparativa de Tiempos
    if time_py > 0 and time_doc > 0:
        print(f"\n--- RESUMEN DE RENDIMIENTO ---")
        print(f"PyMuPDF (Tradicional): {time_py:.4f}s")
        print(f"Docling (IA):         {time_doc:.2f}s")
        print(f"Factor de velocidad:   {time_doc/time_py:.1f}x más rápido PyMuPDF")
else:
    print(f"❌ Error: No se encontró el archivo en {PDF_PATH}")

## 4. Contraste de Resultados
Visualiza cómo cada herramienta interpreta el mismo contenido.

In [ ]:
print("### [TRADICIONAL] SALIDA PYMUPDF (Fragmento) ###")
if text_py:
    print(text_py[:1500] + "...")
else:
    print("(No se pudo extraer texto con PyMuPDF)")

print("\n" + "="*60 + "\n")

print("### [IA] SALIDA DOCLING (Fragmento Markdown) ###")
if text_doc:
    display(Markdown(text_doc[:1500] + "..."))
else:
    print("(No se pudo extraer texto con Docling)")